**Table of contents**<a id='toc0_'></a>    
- [Example fluorescence trajectories at 0.025 kW/cm² - 4 fluorophores, 3 nm, no OET](#toc1_)    
  - [Data generation](#toc1_1_)    
    - [Simulation](#toc1_1_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Example fluorescence trajectories at 0.025 kW/cm² - 4 fluorophores, 3 nm, no OET](#toc0_)

In [ ]:
import warnings

import numpy as np
import pandas as pd

import fluopy.emissions as em
import fluopy.fluorophores as fl
import fluopy.routines as rt
import fluopy.simulation as si
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

saving_at = r"D:\python_output\Chapter_I\1_11_multi_f_et_irradiance\OET\irradiance1"

## <a id='toc1_1_'></a>[Data generation](#toc0_)

### <a id='toc1_1_1_'></a>[Simulation](#toc0_)

In [ ]:
PARAMS_ADJ = {
    k: v
    for k, v in rt.PARAMS_DSTORM.items()
    if k not in ["irradiance", "energy_transfer_parameters"]
}
irradiance = rt.PARAMS_DSTORM["irradiance"] * 0.01
fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=3, count=4, shape="square"
)
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
transitions = fluorophore_system.load_transitions(
    bleaching=True,
    energy_transfer=True,
    **PARAMS_ADJ,
    summarize=True,
    irradiance=irradiance,
    energy_transfer_parameters={"exclude": ["s0", "off"]},
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set.finalize()

rng = np.random.default_rng(1)
for i in range(6):
    simulation = si.Simulation(transition_set)
    simulation.run(
        size=1e6,
        end_time=100,
        seed=rng,
        use_memmap=r"C:\Users\vie43sq\Desktop\Simulations\memmaps\run_7",
    )
    emis = em.Emissions(**rt.PARAMS_EMIS, seed=rng)
    emis.extract(simulation)
    rt.emission_post_processing(emis=emis, seed=rng)
    emis.event_time_series.name = i
    if i == 0:
        df = emis.event_time_series
    else:
        df = pd.concat([df, emis.event_time_series], axis=1, ignore_index=False)
output_file = rf"{saving_at}\emission_df.parquet"
df.to_parquet(output_file)